In [31]:
# Instalação de Dependências e Importações
%pip install -q pandas pyarrow boto3 sqlalchemy psycopg2-binary

import io
import pandas as pd
import boto3
from sqlalchemy import create_engine

# Configuracoes de Infraestrutura (Rede interna do Docker)
config_db = {"host": "postgres", "database": "postgres", "user": "postgres", "password": "senha123", "port": 5432}
config_s3 = {"endpoint_url": "http://minio:9000", "aws_access_key_id": "root-minio", "aws_secret_access_key": "root12345678", "region_name": "local"}

bucket_name = "raw"
destino_s3 = "python_native/ledger_parquet/ledger_analytics.parquet"

Note: you may need to restart the kernel to use updated packages.


In [32]:
# Extração (Extract) via SQLAlchemy e Pandas

print("Criando engine de conexao via SQLAlchemy...")
uri_conexao = f"postgresql+psycopg2://{config_db['user']}:{config_db['password']}@{config_db['host']}:{config_db['port']}/{config_db['database']}"
engine = create_engine(uri_conexao)

print("Conectando e populando DataFrame Pandas em memoria RAM...")
df_financeiro = pd.read_sql_query("SELECT * FROM transacoes_financeiras;", engine)

print(f"DataFrame carregado com sucesso. Formato da tabela: {df_financeiro.shape}")

Criando engine de conexao via SQLAlchemy...
Conectando e populando DataFrame Pandas em memoria RAM...
DataFrame carregado com sucesso. Formato da tabela: (20000, 9)


In [33]:
# Tratamento de Tipos e Rotação Colunar (Código)

print("Tratando colunas UUID para compatibilidade com PyArrow...")
df_financeiro['id_transacao'] = df_financeiro['id_transacao'].astype(str)

print("Rotacionando para formato colunar e aplicando compressao Snappy no Buffer...")
buffer_binario = io.BytesIO()
df_financeiro.to_parquet(buffer_binario, engine='pyarrow', compression='snappy', index=False)
print("Serializacao em lote colunar concluida no buffer binario.")

Tratando colunas UUID para compatibilidade com PyArrow...
Rotacionando para formato colunar e aplicando compressao Snappy no Buffer...
Serializacao em lote colunar concluida no buffer binario.


In [34]:
# Carga (Load) para o Object Storage

print(f"Enviando bytes comprimidos para s3://{bucket_name}/{destino_s3}...")
s3_client = boto3.client('s3', **config_s3)
s3_client.put_object(
    Bucket=bucket_name,
    Key=destino_s3,
    Body=buffer_binario.getvalue(),
    ContentType='application/octet-stream'
)

print("Ingestao Parquet para o MinIO concluida com sucesso!")

Enviando bytes comprimidos para s3://raw/python_native/ledger_parquet/ledger_analytics.parquet...
Ingestao Parquet para o MinIO concluida com sucesso!


In [ ]:
# Encerramento das Conexoes

print("Desalocando o pool de conexoes do SQLAlchemy...")
engine.dispose()
print("Engine desativada com seguranca. Recursos liberados.")

Desalocando o pool de conexoes do SQLAlchemy...
Engine desativada com seguranca. Recursos liberados.


### Analise de Arquitetura: O Formato Parquet no Python Nativo

**Caracteristicas Principais:** Formato binario, autodescritivo e estritamente orientado a colunas (Columnar).

#### Vantagens
* **Alta Compressao:** Agrupar dados por tipo na mesma coluna permite indices de compressao eficientes usando Snappy.
* **Performance Analitica (OLAP):** Motores de busca escaneiam apenas as colunas solicitadas nas queries, ignorando o resto do arquivo e reduzindo o I/O de rede.

#### Desvantagens
* **Custo de Conversao:** Exige processamento e uso de memoria RAM para reorganizar as linhas em colunas no dataframe antes da gravacao.
* **Ilegibilidade:** Arquivo binario compactado nao legivel por humanos sem ferramentas apropriadas.

#### Caso de Uso no Mercado
* Padrao para armazenamento definitivo nas camadas Silver e Gold de Data Lakes e Lakehouses analiticos.